# Multi-Route Model v1.1

This notebook consolidates the best model configuration from feature exploration (7a-7d).

**Final Feature Set:**
- **Lag:** delay_rate_lag1, delay_rate_gradient
- **Weather:** rainy_days_arr_exp, temp_volatility_total_exp, extreme_weather_days_total
- **Holidays:** n_public_holidays_total, pct_school_holiday
- **Encoding:** airline one-hot, route one-hot, month_sin, month_cos
- **Volume:** sectors_scheduled

**Feature Decisions (from 7a-7d):**
- included: `extreme_weather_days_total` - improves model (from 7c)
- included: `n_public_holidays_total`, `pct_school_holiday` - retained holiday features
- excluded: `n_public_holidays_national` - redundant with total (from 7a)
- excluded: `has_major_holiday`, `school_holiday_days` - removed to simplify, low feature importance
- excluded `route_distance_km` - captured by route encoding (from 7b)
- excluded `delay_rate_acceleration` - minimal improvement, requires lag3 (from 7d)

**Data Filtering:**
- Based on notebook notebook 6d, airline-routes with low flight volume <50 flights/month are excluded
- Based on notebook notebook 6e, anomalous route Melbourne - Hobart (year 2019 specifically, which is used as the test dataset) is also excluded. Future work might attempt to exclude year 2019 from test dataset of this route.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date
from calendar import monthrange
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
    confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

import holidays

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed")

%matplotlib inline

## 1. Data Preparation

In [ ]:
# Load multi-route data
df = pd.read_csv('../data/processed/ml_training_data_multiroute_hols.csv')

df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print(f"Original shape: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")

## 2. Filter Low-Volume Airline-Routes

In [ ]:
volume_threshold = 40
airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean().reset_index()
airline_route_volume.columns = ['airline_route', 'avg_volume']
high_volume_ar = airline_route_volume[airline_route_volume['avg_volume'] >= volume_threshold]['airline_route'].tolist()

# Show excluded routes
excluded = airline_route_volume[airline_route_volume['avg_volume'] < volume_threshold].sort_values('avg_volume')
print(f"Volume threshold: {volume_threshold} flights/month")
print(f"\nExcluded airline-routes ({len(excluded)}):")
for _, row in excluded.iterrows():
    print(f"  {row['airline_route']}: {row['avg_volume']:.0f} flights/mo")

df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()
print(f"\nRecords before filtering: {len(df)}")
print(f"Records after filtering:  {len(df_filtered)}")

In [ ]:
# Exclude Melbourne-Hobart route & Adelaide-Sydney (found after running this notebook)
# Reason: Anomalous 2019 data causes unreliable predictions (see notebook 6c)
anomalous_routes = ['Melbourne_Hobart', 'Adelaide_Sydney', 'Perth_Brisbane']
records_before = len(df_filtered)
df_filtered = df_filtered[~df_filtered['route'].isin(anomalous_routes)].copy()
print(f"\nExcluded anomalous routes: {anomalous_routes}")
print(f"Records before: {records_before}")
print(f"Records after:  {len(df_filtered)}")

## 3. Feature Engineering

In [ ]:
# Lagged features
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']

# Cyclical month encoding
df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

# One-hot encoding
airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = list(route_dummies.columns)

# Weather features with transformations
df_filtered['rainy_days_arr_exp'] = np.exp(df_filtered['rainy_days_arr'] / df_filtered['rainy_days_arr'].max())
df_filtered['temp_volatility_total'] = df_filtered['temp_volatility_dep'] + df_filtered['temp_volatility_arr']
df_filtered['temp_volatility_total_exp'] = np.exp(df_filtered['temp_volatility_total'] / df_filtered['temp_volatility_total'].max())
df_filtered['extreme_weather_days_total'] = df_filtered['extreme_weather_days_dep'] + df_filtered['extreme_weather_days_arr']

# Drop NaN (from lag features)
df_clean = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).copy()
print(f"Rows after dropping NaN: {len(df_clean)}")

## 4. Train/Validation/Test Split

In [ ]:
#train_mask = ((df_clean['year'] <= 2017) | (df_clean['year'] == 2024))
train_mask = (((df_clean['year'] >=2010) & (df_clean['year'] <= 2017)) | (df_clean['year'] == 2023))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2024))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print(f"Train (2010-2017, 2024): {train_mask.sum()} samples")
print(f"Validation (2018, 2023): {val_mask.sum()} samples")
print(f"Test (2019, 2025):       {test_mask.sum()} samples")

In [ ]:
# Define final feature set
base_features = airline_cols + route_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']
weather_features = ['rainy_days_arr_exp', 'delay_rate_gradient', 'temp_volatility_total_exp', 'extreme_weather_days_total']
holiday_features = ['n_public_holidays_total', 'pct_school_holiday']

FINAL_FEATURES = base_features + weather_features + holiday_features

print(f"Total features: {len(FINAL_FEATURES)}")
print(f"\nFeature breakdown:")
print(f"  - Airline encoding: {len(airline_cols)}")
print(f"  - Route encoding:   {len(route_cols)}")
print(f"  - Month encoding:   2 (sin, cos)")
print(f"  - Lag features:     2 (lag1, gradient)")
print(f"  - Volume:           1 (sectors_scheduled)")
print(f"  - Weather:          3 (rainy, temp_volatility, extreme)")
print(f"  - Holidays:         2 (public, school)")

In [ ]:
# Prepare data
X_train = df_clean.loc[train_mask, FINAL_FEATURES].values
X_val = df_clean.loc[val_mask, FINAL_FEATURES].values
X_test = df_clean.loc[test_mask, FINAL_FEATURES].values

y_train_reg = df_clean.loc[train_mask, 'delay_rate'].values
y_val_reg = df_clean.loc[val_mask, 'delay_rate'].values
y_test_reg = df_clean.loc[test_mask, 'delay_rate'].values

y_train_clf = df_clean.loc[train_mask, 'is_high_delay'].values
y_val_clf = df_clean.loc[val_mask, 'is_high_delay'].values
y_test_clf = df_clean.loc[test_mask, 'is_high_delay'].values

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Data prepared.")
print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

## 5. Regression Models

In [ ]:
reg_results = []

# Ridge Regression
print("Training Ridge Regression...")
ridge = Ridge(alpha=100)
ridge.fit(X_train_scaled, y_train_reg)

ridge_val_pred = ridge.predict(X_val_scaled)
ridge_test_pred = ridge.predict(X_test_scaled)

ridge_val_r2 = r2_score(y_val_reg, ridge_val_pred)
ridge_test_r2 = r2_score(y_test_reg, ridge_test_pred)
ridge_test_rmse = np.sqrt(mean_squared_error(y_test_reg, ridge_test_pred))
ridge_test_mae = mean_absolute_error(y_test_reg, ridge_test_pred)

print(f"  Val R²:   {ridge_val_r2:.4f}")
print(f"  Test R²:  {ridge_test_r2:.4f}")
print(f"  Test RMSE: {ridge_test_rmse:.4f}")
print(f"  Test MAE:  {ridge_test_mae:.4f}")

reg_results.append({'Model': 'Ridge', 'Val_R2': ridge_val_r2, 'Test_R2': ridge_test_r2, 
                    'Test_RMSE': ridge_test_rmse, 'Test_MAE': ridge_test_mae})

In [ ]:
# Random Forest Regression
print("Training Random Forest Regression...")
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_train_reg)

rf_val_pred = rf_reg.predict(X_val)
rf_test_pred = rf_reg.predict(X_test)

rf_val_r2 = r2_score(y_val_reg, rf_val_pred)
rf_test_r2 = r2_score(y_test_reg, rf_test_pred)
rf_test_rmse = np.sqrt(mean_squared_error(y_test_reg, rf_test_pred))
rf_test_mae = mean_absolute_error(y_test_reg, rf_test_pred)

print(f"  Val R²:   {rf_val_r2:.4f}")
print(f"  Test R²:  {rf_test_r2:.4f}")
print(f"  Test RMSE: {rf_test_rmse:.4f}")
print(f"  Test MAE:  {rf_test_mae:.4f}")

reg_results.append({'Model': 'Random Forest', 'Val_R2': rf_val_r2, 'Test_R2': rf_test_r2,
                    'Test_RMSE': rf_test_rmse, 'Test_MAE': rf_test_mae})

## 6. Classification Models

In [ ]:
clf_results = []

# Logistic Regression
print("Training Logistic Regression...")
logreg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train_clf)

logreg_val_pred = logreg.predict(X_val_scaled)
logreg_val_proba = logreg.predict_proba(X_val_scaled)[:, 1]
logreg_test_pred = logreg.predict(X_test_scaled)
logreg_test_proba = logreg.predict_proba(X_test_scaled)[:, 1]

logreg_val_f1 = f1_score(y_val_clf, logreg_val_pred)
logreg_test_f1 = f1_score(y_test_clf, logreg_test_pred)
logreg_test_auc = roc_auc_score(y_test_clf, logreg_test_proba)
logreg_test_precision = precision_score(y_test_clf, logreg_test_pred)
logreg_test_recall = recall_score(y_test_clf, logreg_test_pred)

print(f"  Val F1:       {logreg_val_f1:.4f}")
print(f"  Test F1:      {logreg_test_f1:.4f}")
print(f"  Test AUC:     {logreg_test_auc:.4f}")
print(f"  Test Precision: {logreg_test_precision:.4f}")
print(f"  Test Recall:    {logreg_test_recall:.4f}")

clf_results.append({'Model': 'Logistic', 'Val_F1': logreg_val_f1, 'Test_F1': logreg_test_f1,
                    'Test_AUC': logreg_test_auc, 'Test_Precision': logreg_test_precision, 
                    'Test_Recall': logreg_test_recall})

In [ ]:
# Random Forest Classification
print("Training Random Forest Classification...")
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train_clf)

rf_clf_val_pred = rf_clf.predict(X_val)
rf_clf_val_proba = rf_clf.predict_proba(X_val)[:, 1]
rf_clf_test_pred = rf_clf.predict(X_test)
rf_clf_test_proba = rf_clf.predict_proba(X_test)[:, 1]

rf_clf_val_f1 = f1_score(y_val_clf, rf_clf_val_pred)
rf_clf_test_f1 = f1_score(y_test_clf, rf_clf_test_pred)
rf_clf_test_auc = roc_auc_score(y_test_clf, rf_clf_test_proba)
rf_clf_test_precision = precision_score(y_test_clf, rf_clf_test_pred)
rf_clf_test_recall = recall_score(y_test_clf, rf_clf_test_pred)

print(f"  Val F1:       {rf_clf_val_f1:.4f}")
print(f"  Test F1:      {rf_clf_test_f1:.4f}")
print(f"  Test AUC:     {rf_clf_test_auc:.4f}")
print(f"  Test Precision: {rf_clf_test_precision:.4f}")
print(f"  Test Recall:    {rf_clf_test_recall:.4f}")

clf_results.append({'Model': 'Random Forest', 'Val_F1': rf_clf_val_f1, 'Test_F1': rf_clf_test_f1,
                    'Test_AUC': rf_clf_test_auc, 'Test_Precision': rf_clf_test_precision,
                    'Test_Recall': rf_clf_test_recall})

In [ ]:
# XGBoost Classification
if HAS_XGB:
    print("Training XGBoost Classification...")
    xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, 
                                 min_child_weight=5, random_state=42, n_jobs=-1)
    xgb_clf.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
    
    xgb_val_pred = xgb_clf.predict(X_val)
    xgb_val_proba = xgb_clf.predict_proba(X_val)[:, 1]
    xgb_test_pred = xgb_clf.predict(X_test)
    xgb_test_proba = xgb_clf.predict_proba(X_test)[:, 1]
    
    xgb_val_f1 = f1_score(y_val_clf, xgb_val_pred)
    xgb_test_f1 = f1_score(y_test_clf, xgb_test_pred)
    xgb_test_auc = roc_auc_score(y_test_clf, xgb_test_proba)
    xgb_test_precision = precision_score(y_test_clf, xgb_test_pred)
    xgb_test_recall = recall_score(y_test_clf, xgb_test_pred)
    
    print(f"  Val F1:       {xgb_val_f1:.4f}")
    print(f"  Test F1:      {xgb_test_f1:.4f}")
    print(f"  Test AUC:     {xgb_test_auc:.4f}")
    print(f"  Test Precision: {xgb_test_precision:.4f}")
    print(f"  Test Recall:    {xgb_test_recall:.4f}")
    
    clf_results.append({'Model': 'XGBoost', 'Val_F1': xgb_val_f1, 'Test_F1': xgb_test_f1,
                        'Test_AUC': xgb_test_auc, 'Test_Precision': xgb_test_precision,
                        'Test_Recall': xgb_test_recall})

## 7. Results Summary

In [ ]:
reg_df = pd.DataFrame(reg_results)
clf_df = pd.DataFrame(clf_results)

print("="*80)
print("REGRESSION RESULTS")
print("="*80)
print(f"\n{'Model':<15} {'Val R²':>10} {'Test R²':>10} {'Test RMSE':>12} {'Test MAE':>10}")
print("-" * 60)
for _, row in reg_df.iterrows():
    print(f"{row['Model']:<15} {row['Val_R2']:>10.4f} {row['Test_R2']:>10.4f} {row['Test_RMSE']:>12.4f} {row['Test_MAE']:>10.4f}")

print("\n" + "="*80)
print("CLASSIFICATION RESULTS")
print("="*80)
print(f"\n{'Model':<15} {'Val F1':>10} {'Test F1':>10} {'Test AUC':>10} {'Precision':>10} {'Recall':>10}")
print("-" * 70)
for _, row in clf_df.iterrows():
    print(f"{row['Model']:<15} {row['Val_F1']:>10.4f} {row['Test_F1']:>10.4f} {row['Test_AUC']:>10.4f} {row['Test_Precision']:>10.4f} {row['Test_Recall']:>10.4f}")

## 8. Feature Importance

In [ ]:
# Feature importance from Random Forest
importance_df = pd.DataFrame({
    'Feature': FINAL_FEATURES,
    'Importance': rf_reg.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (Random Forest Regression):")
print("-" * 50)
for i, row in importance_df.head(15).iterrows():
    print(f"  {row['Feature']:<35} {row['Importance']:.4f}")

In [ ]:
# Visualize top features
fig, ax = plt.subplots(figsize=(10, 6))

top_features = importance_df.head(15)
y_pos = np.arange(len(top_features))

ax.barh(y_pos, top_features['Importance'], color='steelblue', alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_features['Feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Top 15 Feature Importances (Random Forest)')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 9. Performance by Route

Create a new dataframe to assess route-by-route performance.
- Naive (mean): the perdicted delay rate is assumed to be the historical mean (time-averaged)

In [ ]:
# Get predictions for test set
df_test = df_clean[test_mask].copy()
df_test['ridge_pred'] = ridge.predict(X_test_scaled)
df_test['rf_pred'] = rf_reg.predict(X_test)

# Naive baselines:
# 1. Lag1: predict this month = last month (trivial heuristic)
# 2. Mean: predict this month = training set mean (what R² compares against)
df_test['naive_lag1'] = df_test['delay_rate_lag1']
train_mean = y_train_reg.mean()
df_test['naive_mean'] = train_mean

print("Performance by Route (Ridge vs Naive Baselines):")
print("=" * 90)
print(f"{'Route':<18} {'Ridge R²':>10} {'Lag1 R²':>10} {'Mean R²':>10} {'vs Lag1':>10} {'vs Mean':>10}")
print("-" * 90)

route_results = []
for route in sorted(df_test['route'].unique()):
    route_data = df_test[df_test['route'] == route]
    r2 = r2_score(route_data['delay_rate'], route_data['ridge_pred'])
    lag1_r2 = r2_score(route_data['delay_rate'], route_data['naive_lag1'])
    mean_r2 = r2_score(route_data['delay_rate'], route_data['naive_mean'])
    rmse = np.sqrt(mean_squared_error(route_data['delay_rate'], route_data['ridge_pred']))
    mae = mean_absolute_error(route_data['delay_rate'], route_data['ridge_pred'])
    n = len(route_data)
    
    delta_lag1 = r2 - lag1_r2
    delta_mean = r2 - mean_r2  # This should equal r2 since mean_r2 ≈ 0 by definition
    
    print(f"{route:<18} {r2:>10.4f} {lag1_r2:>10.4f} {mean_r2:>10.4f} {delta_lag1:>+10.4f} {delta_mean:>+10.4f}")
    route_results.append({
        'Route': route, 'R2': r2, 'Lag1_R2': lag1_r2, 'Mean_R2': mean_r2,
        'RMSE': rmse, 'MAE': mae, 'n': n
    })

route_df = pd.DataFrame(route_results)

# Overall baselines
overall_lag1_r2 = r2_score(df_test['delay_rate'], df_test['naive_lag1'])
overall_mean_r2 = r2_score(df_test['delay_rate'], df_test['naive_mean'])
print("-" * 90)
print(f"{'OVERALL':<18} {ridge_test_r2:>10.4f} {overall_lag1_r2:>10.4f} {overall_mean_r2:>10.4f} {ridge_test_r2 - overall_lag1_r2:>+10.4f} {ridge_test_r2 - overall_mean_r2:>+10.4f}")

print(f"""
NOTE ON BASELINES:
  - Mean R² ≈ 0 by definition (R² measures improvement over predicting the mean)
  - Lag1 R² shows how well "this month = last month" performs as a heuristic
  - "vs Lag1" shows whether the model adds value beyond trivial persistence
  - "vs Mean" ≈ Ridge R² (confirms R² interpretation)
""")

In [ ]:
# Visualize route performance: Ridge vs Baselines
fig, axes = plt.subplots(2, 1, figsize=(12, 10))
palette = sns.color_palette("Set2",3)

routes = route_df['Route'].values
x = np.arange(len(routes))
width = 0.25

# Left plot: R² comparison (Ridge vs Lag1 vs Mean)
ax = axes[0]
r2_values = route_df['R2'].values
lag1_r2_values = route_df['Lag1_R2'].values
mean_r2_values = route_df['Mean_R2'].values

ax.bar(x - width, r2_values, width, label='Ridge', color=palette[0], alpha=0.8)
ax.bar(x, lag1_r2_values, width, label='Naive (lag1)', color=palette[1], alpha=0.8)
ax.bar(x + width, mean_r2_values, width, label='Naive (mean)', color=palette[2], alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels([r.replace('_', '→') for r in routes], rotation=45, ha='right')
ax.set_ylabel(r'$R^2$')
ax.set_title('Model vs Baselines by Route')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

# Right plot: Improvement over lag1 baseline
ax = axes[1]
improvements_lag1 = route_df['R2'].values - route_df['Lag1_R2'].values
colors = ['#27ae60' if imp > 0 else '#e74c3c' for imp in improvements_lag1]

ax.bar(x, improvements_lag1, color=colors, alpha=0.8)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([r.replace('_', '→') for r in routes], rotation=45, ha='right')
ax.set_ylabel(r'$\Delta R^2$')
ax.set_title('Improvement Over Lag1 Baseline\n(Marginal value of model)')
ax.grid(True, alpha=0.3, axis='y')


plt.tight_layout()
plt.show()

# Print interpretation
print("INTERPRETATION:")
print("-" * 70)
print(f"  Mean baseline R² ≈ 0 (by definition of R²)")
print(f"  Lag1 baseline R² = {overall_lag1_r2:.4f} (strong autocorrelation)")
print(f"  Ridge model R²   = {ridge_test_r2:.4f}")
print(f"")
print(f"  The lag1 baseline already explains {overall_lag1_r2*100:.1f}% of variance.")
print(f"  The Ridge model adds {(ridge_test_r2 - overall_lag1_r2)*100:.1f} percentage points.")
print(f"  This means {(ridge_test_r2 - overall_lag1_r2) / ridge_test_r2 * 100:.1f}% of the model's R² comes from features beyond lag1.")

## 10. Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Ridge
ax = axes[0]
ax.scatter(y_test_reg, ridge_test_pred, alpha=0.4, s=20)
ax.plot([0, 0.6], [0, 0.6], 'r--', label='Perfect prediction')
ax.set_xlabel('Actual Delay Rate')
ax.set_ylabel('Predicted Delay Rate')
ax.set_title(f'Ridge Regression (R² = {ridge_test_r2:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.6)
ax.set_ylim(0, 0.6)

# Random Forest
ax = axes[1]
ax.scatter(y_test_reg, rf_test_pred, alpha=0.4, s=20)
ax.plot([0, 0.6], [0, 0.6], 'r--', label='Perfect prediction')
ax.set_xlabel('Actual Delay Rate')
ax.set_ylabel('Predicted Delay Rate')
ax.set_title(f'Random Forest (R² = {rf_test_r2:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.6)
ax.set_ylim(0, 0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix Comparison for Classification Models
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Get confusion matrices
cm_logreg = confusion_matrix(y_test_clf, logreg_test_pred)
cm_rf = confusion_matrix(y_test_clf, rf_clf_test_pred)
cm_xgb = confusion_matrix(y_test_clf, xgb_test_pred) if HAS_XGB else None

# Labels
labels = ['Normal\n(≤25%)', 'High Delay\n(>25%)']

# Plot function
def plot_cm(ax, cm, title, f1, auc):
    im = ax.imshow(cm, cmap='Blues', aspect='auto')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{title}\nF1={f1:.3f}, AUC={auc:.3f}')
    
    # Add text annotations
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, f'{cm[i, j]}', ha='center', va='center', color=color, fontsize=14)
    
    return im

# Plot each model
plot_cm(axes[0], cm_logreg, 'Logistic Regression', logreg_test_f1, logreg_test_auc)
plot_cm(axes[1], cm_rf, 'Random Forest', rf_clf_test_f1, rf_clf_test_auc)
if HAS_XGB:
    plot_cm(axes[2], cm_xgb, 'XGBoost', xgb_test_f1, xgb_test_auc)
else:
    axes[2].axis('off')
    axes[2].text(0.5, 0.5, 'XGBoost not available', ha='center', va='center')

plt.tight_layout()
plt.show()

# Print summary
print("Confusion Matrix Summary (Test Set):")
print("=" * 70)
print(f"{'Model':<18} {'TN':>8} {'FP':>8} {'FN':>8} {'TP':>8} {'Accuracy':>10}")
print("-" * 70)

tn, fp, fn, tp = cm_logreg.ravel()
acc = (tn + tp) / (tn + fp + fn + tp)
print(f"{'Logistic':<18} {tn:>8} {fp:>8} {fn:>8} {tp:>8} {acc:>10.4f}")

tn, fp, fn, tp = cm_rf.ravel()
acc = (tn + tp) / (tn + fp + fn + tp)
print(f"{'Random Forest':<18} {tn:>8} {fp:>8} {fn:>8} {tp:>8} {acc:>10.4f}")

if HAS_XGB:
    tn, fp, fn, tp = cm_xgb.ravel()
    acc = (tn + tp) / (tn + fp + fn + tp)
    print(f"{'XGBoost':<18} {tn:>8} {fp:>8} {fn:>8} {tp:>8} {acc:>10.4f}")

print(f"""
Legend:
  TN = True Negatives  (correctly predicted normal)
  FP = False Positives (incorrectly predicted high delay)
  FN = False Negatives (missed high delay - worst outcome)
  TP = True Positives  (correctly predicted high delay)
""")

## 12. Final Summary

In [ ]:
print("="*80)
print("FINAL MODEL SUMMARY")
print("="*80)

print(f"""
FEATURE SET ({len(FINAL_FEATURES)} features):
  - Lag: delay_rate_lag1, delay_rate_gradient
  - Weather: rainy_days_arr_exp, temp_volatility_total_exp, extreme_weather_days_total
  - Holidays: n_public_holidays_total, pct_school_holiday
  - Encoding: {len(airline_cols)} airline dummies, {len(route_cols)} route dummies, month_sin/cos
  - Volume: sectors_scheduled

DATA FILTERING:
  - Excluded airline-routes with <50 flights/month
  - Excluded anomalous routes: {', '.join(anomalous_routes)}
  - Train: 2010-2017 + 2024 ({train_mask.sum()} samples)
  - Val: 2018 + 2023 ({val_mask.sum()} samples)
  - Test: 2019 + 2025 ({test_mask.sum()} samples)

REGRESSION PERFORMANCE (Test):
  Ridge:         R² = {ridge_test_r2:.4f}, RMSE = {ridge_test_rmse:.4f}
  Random Forest: R² = {rf_test_r2:.4f}, RMSE = {rf_test_rmse:.4f}

CLASSIFICATION PERFORMANCE (Test):
  Logistic:      F1 = {logreg_test_f1:.4f}, AUC = {logreg_test_auc:.4f}
  Random Forest: F1 = {rf_clf_test_f1:.4f}, AUC = {rf_clf_test_auc:.4f}""")

if HAS_XGB:
    print(f"  XGBoost:       F1 = {xgb_test_f1:.4f}, AUC = {xgb_test_auc:.4f}")

# Best route/worst route
best_route = route_df.loc[route_df['R2'].idxmax()]
worst_route = route_df.loc[route_df['R2'].idxmin()]

print(f"""
ROUTE PERFORMANCE (Ridge):
  Best:  {best_route['Route']} (R² = {best_route['R2']:.4f})
  Worst: {worst_route['Route']} (R² = {worst_route['R2']:.4f})
""")

### Observations

- delay_rate_lag1 is the dominant predictor (previous month's delay rate)
- Volume filtering improves predictions by removing noisy low-volume routes
- Simple models (Ridge) perform nearly as well as complex models (RF, XGBoost)

## 13. Next Step

Since it has been the models' hyperparameters have not been tuned for the multi-route data, this will be done in the next notebook.